In [ ]:
import os
import json
import re
import requests
import jsonschema
from jsonschema import validate

# =====================================================================
# 1. API ENVIRONMENT SETUP & CONNECTIVITY ENGINE
# =====================================================================

# Initialize the API Key in the environment to avoid hardcoding credentials
os.environ['LLM_API_KEY'] = 'your_actual_public_api_key_here'
API_KEY = os.environ.get('LLM_API_KEY')

# Public Routing Gateway API Endpoint Configuration
LLM_API_URL = "https://openrouter.ai"
MODEL_NAME = "google/gemini-2.5-flash"  # Using a highly reliable instruction-following JSON model

def call_llm(system_prompt: str, user_prompt: str, temperature: float = 0.0, max_tokens: int = 512) -> str:
    """Executes a structured POST request to an OpenAI-compatible JSON API endpoint."""
    if not API_KEY or "your_actual_public_api_key" in API_KEY:
        print("[CONFIGURATION WARNING] Valid API key must be exported to environment.")
        return None
        
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }
    
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    
    try:
        response = requests.post(LLM_API_URL, headers=headers, json=payload, timeout=20)
        if response.status_code != 200:
            print(f"[API HTTP ERROR] Code: {response.status_code} | Body: {response.text}")
            return None
        
        data = response.json()
        return data['choices'][0]['message']['content']
    except Exception as e:
        print(f"[PIPELINE CONNECTION ERROR] Request block aborted: {str(e)}")
        return None

# =====================================================================
# 2. PROMPT ARCHITECTURE AND TARGET VALIDATION SCHEMA
# =====================================================================

# Target validation schema containing exactly 5 scalar properties
SCORING_SCHEMA = {
    "type": "object",
    "properties": {
        "risk_tier": {"type": "string", "enum": ["low", "medium", "high"]},
        "flag_for_review": {"type": "boolean"},
        "primary_signal": {"type": "string"},
        "confidence": {"type": "string", "enum": ["low", "medium", "high"]},
        "recommended_action": {"type": "string", "enum": ["approve", "manual_underwrite", "decline"]}
    },
    "required": ["risk_tier", "flag_for_review", "primary_signal", "confidence", "recommended_action"],
    "additionalProperties": False
}

SYSTEM_PROMPT = """You are an expert financial risk underwriting system. Your role is to evaluate a single dataset record structured as a JSON object against a strict business rubric and output a structured assessment.

Scoring Rubric Criteria:
1. risk_tier: Assign 'high' if credit_score < 600 or debt_to_income > 0.45. Assign 'medium' if credit_score is between 600 and 680, or debt_to_income is between 0.35 and 0.45. Otherwise, assign 'low'.
2. flag_for_review: Set to true if risk_tier is 'high' or if late_payments > 0. Set to false otherwise.
3. primary_signal: A brief single-sentence string identifying the strongest statistical driver of the assigned risk tier.
4. confidence: Assign 'high' if active_accounts >= 3. Otherwise, assign 'medium' or 'low'.
5. recommended_action: Map strictly to: 'decline' for high risk, 'manual_underwrite' for medium risk or flagged items, and 'approve' for low-risk unflagged items.

Output Format:
You must output ONLY a valid JSON object matching the requested fields. Do not include markdown formatting blocks (such as ```json), backticks, or any conversational filler text.

Worked Input-Output Example:
Input: {"credit_score": 580, "debt_to_income": 0.38, "late_payments": 1, "active_accounts": 5}
Output: {"risk_tier": "high", "flag_for_review": true, "primary_signal": "Credit score falls below the high-risk threshold of 600.", "confidence": "high", "recommended_action": "decline"}"""

# =====================================================================
# 3. SAFETY GUARDRAILS AND INTEGRATED PARSING ENGINE
# =====================================================================

def has_pii(text: str) -> bool:
    """Scans structural string payload inputs for emails or 10-digit phone layouts."""
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'
    return bool(re.search(email_pattern, text) or re.search(phone_pattern, text))

def run_scoring_pipeline(record_dict: dict, temp: float = 0.0) -> tuple:
    """Protects endpoints, parses string inputs, and runs strict JSON validation."""
    user_input_str = json.dumps(record_dict)
    
    # Evaluate safety filter layer
    if has_pii(user_input_str):
        print("Input blocked: PII detected.")
        return None, "BLOCKED_BY_GUARDRAIL"
        
    # Execute inference query
    raw_response = call_llm(SYSTEM_PROMPT, user_input_str, temperature=temp)
    if not raw_response:
        return get_fallback_value("API Connection Terminated")
        
    clean_json_str = raw_response.strip()
    
    # Structural Parsing and Schema Verification Engine
    try:
        parsed_data = json.loads(clean_json_str)
        validate(instance=parsed_data, schema=SCORING_SCHEMA)
        return parsed_data, "PASS"
    except json.JSONDecodeError as jde:
        error_context = f"JSON Decode Failure: {str(jde)}"
    except jsonschema.ValidationError as jve:
        error_context = f"Schema Validation Failure: {str(jve.message)}"
        
    return get_fallback_value(error_context)

def get_fallback_value(error_log_msg: str) -> tuple:
    """Returns a completely nullified data dictionary when system faults occur."""
    fallback_dict = {
        "risk_tier": None,
        "flag_for_review": None,
        "primary_signal": f"Pipeline Error Triggered Fallback: {error_log_msg}",
        "confidence": None,
        "recommended_action": None
    }
    return fallback_dict, f"FAIL ({error_log_msg})"

# =====================================================================
# 4. END-TO-END PIPELINE VALIDATION CHECKS
# =====================================================================

print("--- TASK 1: VERIFYING CONTEXT ECHO ENGINE ---")
echo_check = call_llm("You are a single-word responder.", "Reply with only the word: hello", temperature=0.0)
print(f"Sanity Check Output: {echo_check}\n")


print("--- TASK 2: TESTING PII SAFETY GUARDRAILS ---")
pii_unsafe_record = {"credit_score": 710, "email": "test_user@domain.com", "debt_to_income": 0.25}
pii_safe_record = {"credit_score": 710, "debt_to_income": 0.25}

print("Injesting Unsafe Record containing Email Address:")
if has_pii(json.dumps(pii_unsafe_record)):
    print("Input blocked: PII detected.")
else:
    print("Passed Safety Filter Engine.")

print("\nInjesting Clean Clean Record:")
if has_pii(json.dumps(pii_safe_record)):
    print("Input blocked: PII detected.")
else:
    print("Passed Safety Filter Engine.")
print("\n")


print("--- TASK 3: COMPREHENSIVE PRODUCTION BATCH SCORING ---")
cleaned_dataset_batch = [
    {"credit_score": 520, "debt_to_income": 0.52, "late_payments": 3, "active_accounts": 4},
    {"credit_score": 640, "debt_to_income": 0.38, "late_payments": 0, "active_accounts": 5},
    {"credit_score": 780, "debt_to_income": 0.22, "late_payments": 0, "active_accounts": 6}
]

for idx, record in enumerate(cleaned_dataset_batch, 1):
    print(f"=== BATCH PROCESSING RECORD {idx} ===")
    print(f"Raw Input Payload: {record}")
    
    # Process using stable deterministic configurations
    validated_json, execution_status = run_scoring_pipeline(record, temp=0.0)
    
    print(f"Pipeline Validation Status: {execution_status}")
    print(f"Clean Parsed JSON Assessment:\n{json.dumps(validated_json, indent=2)}")
    print("=" * 60)